# Testing data load stuff to work out script to prep data for npy load in correct format
- see `scripts/get_npy.py`

In [1]:
import pandas as pd
import os
import pyarrow
import fastparquet
import numpy as np

In [2]:
df = pd.read_parquet(os.path.join("..", "data", "10376_T1_MM_25022022_acc_gyr.parquet"))
print(df.head(5))

       time_gyr  gyr_x  gyr_y  gyr_z      time_acc     acc_x     acc_y  \
0  1.644538e+09   1.26  -5.60  -2.59           NaN       NaN       NaN   
1  1.644538e+09   0.91  -5.53  -2.45  1.644538e+09 -2.327167 -8.944734   
2  1.644538e+09   1.19  -5.18  -2.73  1.644538e+09 -2.327167 -8.944734   
3  1.644538e+09   1.12  -5.60  -2.80  1.644538e+09 -2.317586 -8.963896   
4  1.644538e+09   0.91  -5.67  -2.59  1.644538e+09 -2.279281 -8.935162   

      acc_z tag      date timestamp   p_id  overall_nep_status  gs_flag  
0       NaN  MM  25022022        T1  10376                   0        0  
1 -3.725380  MM  25022022        T1  10376                   0        0  
2 -3.725380  MM  25022022        T1  10376                   0        0  
3 -3.763684  MM  25022022        T1  10376                   0        0  
4 -3.763684  MM  25022022        T1  10376                   0        0  


In [5]:
# 1. Total count of missing values per column
print("--- Missing Value Counts ---")
print(df[['time_gyr', 'acc_x', 'acc_y', 'acc_z']].isna().sum())

# 2. Check for non-finite values (Inf / -Inf)
print("\n--- Non-Finite (Inf) Counts ---")
print((~np.isfinite(df[['acc_x', 'acc_y', 'acc_z']]) & df[['acc_x', 'acc_y', 'acc_z']].notna()).sum())

# 3. Locate exact row indices where missingness occurs
nan_rows = df[df[['time_gyr', 'acc_x', 'acc_y', 'acc_z']].isna().any(axis=1)]
print(f"\nTotal rows with NaNs: {len(nan_rows)}")

if not nan_rows.empty:
    print("\nFirst 5 rows with NaNs:")
    print(nan_rows.head())

    print("\nLast 5 rows with NaNs:")
    print(nan_rows.tail())

    # 4. Check if NaNs are isolated per column or aligned across all columns
    print("\nNaN Row Breakdown:")
    print(f"Rows missing time: {df['time_gyr'].isna().sum()}")
    print(f"Rows missing all acceleration (x, y, z): {df[['acc_x', 'acc_y', 'acc_z']].isna().all(axis=1).sum()}")
    print(f"Rows missing partial acceleration: {(df[['acc_x', 'acc_y', 'acc_z']].isna().sum(axis=1) % 3 != 0).sum()}")

--- Missing Value Counts ---
time_gyr    0
acc_x       1
acc_y       1
acc_z       1
dtype: int64

--- Non-Finite (Inf) Counts ---
acc_x    0
acc_y    0
acc_z    0
dtype: int64

Total rows with NaNs: 1

First 5 rows with NaNs:
       time_gyr  gyr_x  gyr_y  gyr_z  time_acc  acc_x  acc_y  acc_z tag  \
0  1.644538e+09   1.26   -5.6  -2.59       NaN    NaN    NaN    NaN  MM   

       date timestamp   p_id  overall_nep_status  gs_flag  
0  25022022        T1  10376                   0        0  

Last 5 rows with NaNs:
       time_gyr  gyr_x  gyr_y  gyr_z  time_acc  acc_x  acc_y  acc_z tag  \
0  1.644538e+09   1.26   -5.6  -2.59       NaN    NaN    NaN    NaN  MM   

       date timestamp   p_id  overall_nep_status  gs_flag  
0  25022022        T1  10376                   0        0  

NaN Row Breakdown:
Rows missing time: 0
Rows missing all acceleration (x, y, z): 1
Rows missing partial acceleration: 0


In [ ]:
_